In [5]:
import pandas as pd
from pathlib import Path

csv_path = Path("/Users/silmarabarnabe/Desktop/Projetos/vigimed/data/03_gold/dim_via_adm/dim_via_adm.csv")
out_path = csv_path.with_suffix(".parquet")

df = pd.read_csv(csv_path, sep=";", encoding="utf-8")
df.to_parquet(out_path, engine="pyarrow", index=False)
print(f"Linhas: {len(df)}, colunas: {list(df.columns)}")
print(f"Salvo: {out_path} ({out_path.stat().st_size} bytes)")
 

Linhas: 11, colunas: ['VIA_ADMINISTRACAO_CHAVE', 'VIA_ADMINISTRACAO_VALOR', 'VIA_ADMINISTRACAO_DESCRIPTION', 'VIA_ADMINISTRACAO_DESCRIPTION_PT']
Salvo: /Users/silmarabarnabe/Desktop/Projetos/vigimed/data/03_gold/dim_via_adm/dim_via_adm.parquet (3902 bytes)


In [6]:
from pathlib import Path
import os
import subprocess

import pandas as pd
import pyarrow.parquet as pq

# Raiz dos dados gold (mesma lógica do 2.EDA_Artigo)
_data_dir = os.environ.get("VIGIMED_DATA_DIR")
if _data_dir:
    GOLD_DATA_PATH = Path(_data_dir)
else:
    try:
        repo_root = subprocess.check_output(
            ["git", "rev-parse", "--show-toplevel"],
            text=True,
        ).strip()
        GOLD_DATA_PATH = Path(repo_root) / "data" / "03_gold"
    except Exception:
        GOLD_DATA_PATH = Path.cwd().resolve() / ".." / ".." / "data" / "03_gold"

TABLE_PATHS = {
    # dimensões
    "dim_atc": GOLD_DATA_PATH / "dim_atc" / "dim_atc.parquet",
    "dim_concentracao": GOLD_DATA_PATH / "dim_concentracao" / "dim_concentracao.parquet",
    "dim_detentor_registro": GOLD_DATA_PATH / "dim_detentor_registro" / "dim_detentor_registro.parquet",
    "dim_dose": GOLD_DATA_PATH / "dim_dose" / "dim_dose.parquet",
    "dim_dose_frequencia": GOLD_DATA_PATH / "dim_dose_frequencia" / "dim_dose_frequencia.parquet",
    "dim_duracao": GOLD_DATA_PATH / "dim_duracao" / "dim_duracao.parquet",
    "dim_forma_farmaceutica": GOLD_DATA_PATH / "dim_forma_farmaceutica" / "dim_forma_farmaceutica.parquet",
    "dim_notificacoes": GOLD_DATA_PATH / "dim_notificacoes" / "dim_notificacoes.parquet",
    "dim_soc_llt": GOLD_DATA_PATH / "dim_soc_llt" / "dim_soc_llt.parquet",
    "dim_via_adm": GOLD_DATA_PATH / "dim_via_adm" / "dim_via_adm.parquet",
    # fatos
    #"fat_analytical": GOLD_DATA_PATH / "fat_analytical" / "fat_analytical.parquet",
    "fat_medicamentos": GOLD_DATA_PATH / "fat_medicamentos" / "fat_medicamentos.parquet",
    "fat_reacoes": GOLD_DATA_PATH / "fat_reacoes" / "fat_reacoes.parquet",
}

In [7]:
def parquet_stats(path: Path) -> dict:
    """Linhas e colunas via metadados (sem carregar a tabela inteira na RAM)."""
    path = Path(path)
    size_mb = path.stat().st_size / (1024**2)
    pf = pq.ParquetFile(path)
    meta = pf.metadata
    return {
        "arquivo": path.name,
        "caminho": str(path.resolve()),
        "linhas": meta.num_rows,
        "colunas": meta.num_columns,
        "tamanho_mb": round(size_mb, 3),
    }


rows = []
for nome, p in TABLE_PATHS.items():
    if not p.exists():
        rows.append(
            {
                "tabela": nome,
                "arquivo": p.name,
                "linhas": None,
                "colunas": None,
                "tamanho_mb": None,
                "observacao": "arquivo não encontrado",
            }
        )
        continue
    s = parquet_stats(p)
    rows.append(
        {
            "tabela": nome,
            **{k: v for k, v in s.items() if k != "caminho"},
            "observacao": "",
        }
    )

shape_df = pd.DataFrame(rows)
shape_df = shape_df[["tabela", "arquivo", "linhas", "colunas", "tamanho_mb", "observacao"]]
display(shape_df)

GOLD_DATA_PATH

,tabela,arquivo,linhas,colunas,tamanho_mb,observacao
0,dim_atc,dim_atc.parquet,6897,17,0.266,
1,dim_concentracao,dim_concentracao.parquet,10,2,0.002,
2,dim_detentor_registro,dim_detentor_registro.parquet,267,2,0.006,
3,dim_dose,dim_dose.parquet,40,2,0.003,
4,dim_dose_frequencia,dim_dose_frequencia.parquet,11,2,0.002,
5,dim_duracao,dim_duracao.parquet,8,2,0.002,
6,dim_forma_farmaceutica,dim_forma_farmaceutica.parquet,21,2,0.002,
7,dim_notificacoes,dim_notificacoes.parquet,311134,54,66.726,
8,dim_soc_llt,dim_soc_llt.parquet,142785,14,4.561,
9,dim_via_adm,dim_via_adm.parquet,11,4,0.004,


PosixPath('/Users/silmarabarnabe/Desktop/Projetos/vigimed/data/03_gold')